In [1]:
# [CELL 1] STUDENT CONFIGURATION

LAST_NAME = "Rosales"
STUDENT_ID = "TUPM-26-3624"

SEED_DIGIT = 4
ID_SUM = 23
NAME_LENGTH = len(LAST_NAME)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name length:", NAME_LENGTH)

Student: Rosales
Student ID: TUPM-26-3624
Seed Digit: 4
ID Sum: 23
Name length: 7


In [2]:
# [CELL 2] CREATE ACTIVITY WORKSPACE 

from pathlib import Path

WORK_DIR = Path(".")

DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Workspace Created Successfully!")
print("Data Directory:", DATA_DIR)
print("Output Directory:", OUTPUT_DIR)
print("Backup Directory:", BACKUP_DIR)

Workspace Created Successfully!
Data Directory: data
Output Directory: output
Backup Directory: backup


In [4]:
# [CELL 3] INSPECT THE CSV DATASET

import csv 

dataset_path = DATA_DIR / "weblog.csv"

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.reader(file)

    headers = next(reader)
    print("Column names:")
    print(headers)

    print("\nFirst 5 Records:")
    for i, row in enumerate(reader):
        if i == 5:
            break
        print(row)

Column names:
['IP', 'Time', 'URL', 'Staus']

First 5 Records:
['10.128.2.1', '[29/Nov/2017:06:58:55', 'GET /login.php HTTP/1.1', '200']
['10.128.2.1', '[29/Nov/2017:06:59:02', 'POST /process.php HTTP/1.1', '302']
['10.128.2.1', '[29/Nov/2017:06:59:03', 'GET /home.php HTTP/1.1', '200']
['10.131.2.1', '[29/Nov/2017:06:59:04', 'GET /js/vendor/moment.min.js HTTP/1.1', '200']
['10.130.2.1', '[29/Nov/2017:06:59:06', 'GET /bootstrap-3.3.7/js/bootstrap.js HTTP/1.1', '200']


In [6]:
# [CELL 4] CHECK DATASET STRUCTURE

import csv

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    records = list(reader)

print("Total Records:", len(records))
print("\nColumn Names:")
print(reader.fieldnames)

print("\nMissing Values:")
for column in reader.fieldnames:
    missing = sum(
        1 for row in records
        if not row[column].strip()
    )
    print(f"{column}: {missing}")

print("\nFirst 5 Records:")
for row in records[:5]:
    print(row)


Total Records: 16007

Column Names:
['IP', 'Time', 'URL', 'Staus']

Missing Values:
IP: 0
Time: 0
URL: 0
Staus: 0

First 5 Records:
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:58:55', 'URL': 'GET /login.php HTTP/1.1', 'Staus': '200'}
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:59:02', 'URL': 'POST /process.php HTTP/1.1', 'Staus': '302'}
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:59:03', 'URL': 'GET /home.php HTTP/1.1', 'Staus': '200'}
{'IP': '10.131.2.1', 'Time': '[29/Nov/2017:06:59:04', 'URL': 'GET /js/vendor/moment.min.js HTTP/1.1', 'Staus': '200'}
{'IP': '10.130.2.1', 'Time': '[29/Nov/2017:06:59:06', 'URL': 'GET /bootstrap-3.3.7/js/bootstrap.js HTTP/1.1', 'Staus': '200'}


In [7]:
# [CELL 5] SET ANALYSIS PARAMETERS

RECORDS_TO_PROCESS = 1000 + 23
TARGET_STATUS_CLASS = (4 % 4) + 2

print("Records to Process:", RECORDS_TO_PROCESS)
print("Target Status Class:", f"{TARGET_STATUS_CLASS}xx")

Records to Process: 1023
Target Status Class: 2xx


In [12]:
# [CELL 6] PROCESS WEB LOGS

selected_records = []
status_counts = {}
url_counts = {}

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)

    for i, row in enumerate(reader):
        if i >= RECORDS_TO_PROCESS:
            break

        status = row["Staus"].strip()

        if status.isdigit() and int(status) // 100 == TARGET_STATUS_CLASS:
            selected_records.append(row)

            status_counts[status] = status_counts.get(status, 0) + 1
            url = row["URL"].strip()
            url_counts[url] = url_counts.get(url, 0) + 1

print("Records Processed:", min(RECORDS_TO_PROCESS, i + 1))
print("Records in Target Class:", len(selected_records))
print("Counts per Status Code:", status_counts)

Records Processed: 1023
Records in Target Class: 853
Counts per Status Code: {'200': 845, '206': 8}


In [13]:
# [CELL 7] TOP 5 MOST REQUESTED URLS

top_urls = sorted(
    url_counts.items(),
    key=lambda item: item[1],
    reverse=True
)[:5]

print("Top 5 Most Requested URLs:")

for url, count in top_urls:
    print(url, "-", count, "requests")

Top 5 Most Requested URLs:
GET /login.php HTTP/1.1 - 105 requests
GET /home.php HTTP/1.1 - 91 requests
GET /css/bootstrap.min.css HTTP/1.1 - 38 requests
GET /css/normalize.css HTTP/1.1 - 38 requests
GET /js/vendor/jquery-1.12.0.min.js HTTP/1.1 - 37 requests


In [9]:
# [CELL 7] CHECK STATUS VALUES

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)

    for i, row in enumerate(reader):
        if i == 10:
            break
        print(repr(row["Staus"]))

'200'
'302'
'200'
'200'
'200'
'200'
'200'
'200'
'200'
'302'


In [11]:
# [CELL 8] TEST STATUS CLASS

for status in ["200", "302", "404", "500"]:
    if int(status) // 100 == TARGET_STATUS_CLASS:
        print(status, "- MATCH")
    else:
        print(status, "- NOT MATCH")

200 - MATCH
302 - NOT MATCH
404 - NOT MATCH
500 - NOT MATCH


In [14]:
# [CELL 8] CREATE OUPUT FOLDERS

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Output folder ready:", OUTPUT_DIR)
print("Backup folder ready:", BACKUP_DIR)

Output folder ready: output
Backup folder ready: backup


In [17]:
# [CELL 9] EXPORT FILTERED CSV

filtered_csv = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_filtered.csv"

with open(filtered_csv, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["IP", "Time", "URL", "Staus"])
    writer.writeheader()
    writer.writerows(selected_records)

print("Filtered CSV created:", filtered_csv)
print("Records saved:", len(selected_records))

Filtered CSV created: output\Rosales_TUPM-26-3624_filtered.csv
Records saved: 853


In [19]:
# [CELL 10] CREATE JSON SUMMARY

import json

summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": RECORDS_TO_PROCESS,
    "target_status_class": f"{TARGET_STATUS_CLASS}xx",
    "records_in_target_class": len(selected_records),
    "status_counts": status_counts,
    "top_5_urls": [
        {"url": url, "requests": count}
        for url, count in top_urls
    ]
}

json_file = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_summary.json"

with open(json_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON SUmmary created:", json_file)

JSON SUmmary created: output\Rosales_TUPM-26-3624_summary.json


In [20]:
# [CELL 11] BACKUP JSON FILE

import shutil

backup_file = BACKUP_DIR / json_file.name

shutil.copy(json_file, backup_file)

if backup_file.exists():
    print("Backup created successfully!")
    print("Backup file:", backup_file)
    print("Backup verified: YES")
else:
    print("Backup verification: FAILED")

Backup created successfully!
Backup file: backup\Rosales_TUPM-26-3624_summary.json
Backup verified: YES


In [21]:
# [CELL 12] DISPLAU FINAL REPORT

print("WEB SERVER ACCESS LOG ANALYSIS")
print("----------------------------------------------")
print("Student:", LAST_NAME)
print("Records Processed:", RECORDS_TO_PROCESS)
print("Target Status Class:", f"{TARGET_STATUS_CLASS}xx")
print("Records in target Class:", len(selected_records))

print("\nCount per Status Code:")
for status, count in status_counts.items():
    print(status, ":", count)

print("\nTop 5 Most Requested URLs:")
for url, count in top_urls:
    print(url, "-", count, "requests")

print("\nFiltered CSV:", filtered_csv)
print("JSON SUmmary:", json_file)
print("JSON Backup:", backup_file)
print("Backup Verified:", backup_file.exists())

WEB SERVER ACCESS LOG ANALYSIS
----------------------------------------------
Student: Rosales
Records Processed: 1023
Target Status Class: 2xx
Records in target Class: 853

Count per Status Code:
200 : 845
206 : 8

Top 5 Most Requested URLs:
GET /login.php HTTP/1.1 - 105 requests
GET /home.php HTTP/1.1 - 91 requests
GET /css/bootstrap.min.css HTTP/1.1 - 38 requests
GET /css/normalize.css HTTP/1.1 - 38 requests
GET /js/vendor/jquery-1.12.0.min.js HTTP/1.1 - 37 requests

Filtered CSV: output\Rosales_TUPM-26-3624_filtered.csv
JSON SUmmary: output\Rosales_TUPM-26-3624_summary.json
JSON Backup: backup\Rosales_TUPM-26-3624_summary.json
Backup Verified: True


In [22]:
# [CELL 13] SAVE TEXT REPORT

report_file = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_report.txt"

with open(report_file, "w", encoding="utf-8") as file:
    file.write("WEB SERVER ACCESS LOG ANALYSIS\n")
    file.write("----------------------------------------------\n")
    file.write(f"Student: {LAST_NAME}\n")
    file.write(f"Records Processed: {RECORDS_TO_PROCESS}\n")
    file.write(f"Target Status Class: {TARGET_STATUS_CLASS}xx\n")
    file.write(f"Records in target Class: {len(selected_records)}\n")

    file.write("\nCount per Status Code:\n")
    for status, count in status_counts.items():
        file.write(f"{status}: {count}\n")

    file.write("\nTop 5 Most Requested URLs:\n")
    for url, count in top_urls:
        file.write(f"{url} - {count} requests\n")

    file.write(f"\nFiltered CSV: {filtered_csv}\n")
    file.write(f"Json Summary: {json_file}\n")
    file.write(f"JSON Backup: {backup_file}\n")
    file.write(f"Backup Verified: {backup_file.exists()}\n")

print("Text report created:", report_file)

Text report created: output\Rosales_TUPM-26-3624_report.txt


In [ ]:
# [CELL 14] VERIFY OUTPUT FILES

print("FILTERED CSV:", filtered_csv.exists())
print("JSON SUMMARY:", json_file.exists())
print("JSON BACKUP:", backup_file.exists())
print("TEXT REPORT:", report_file.exists())